### Parse Trees

<font size = "4">

Parse trees can be used to represent structures like sentences in natural language processing:

<div style="text-align: center;">
  <img src="files/nlParse.png" alt="Centered image" width = "550">
  <br>
  <br>
  <figcaption>A Parse Tree for a Simple Sentence<br>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>

<br>

<font size = "4">

Or arithmetic expressions:

<div style="text-align: center;">
  <img src="files/meParse.png" alt="Centered image" width = "450">
  <br>
  <br>
  <figcaption>Parse Tree for ((7 + 3) x (5 - 2))<br>
  <font size = "1"> Miller, Randum, Yasinovskyy (Problem Solving with Algorithms and Data Structures using Python)</figcaption>
</div>

<br>


<font size = "4">

- We will implement code to build the arithmetic parse tree, and then code to evaluate it.

- Note that in our ``BinaryTree`` implementation, nodes are only linked *downward* from parent to child, not in the reverse order.

- When we parse the string "((7+3)*(5-2))", we will need to move to the lowest level of the tree and then back up.

- How do we move back up when nodes do not point at their parents?

    1. Adapt the `BinaryTree` class so that children point to parents.

    2. Try to use recursion.

    3. Use an auxiliary data structure, like a stack, and push a tree to the stack when we visit its children.

In [10]:
from datasci531 import Stack, BinaryTree

def build_parse_tree(fp_expr):
    fp_list = fp_expr.split()
    p_stack = Stack()
    expr_tree = BinaryTree("")
    # p_stack.push(expr_tree)
    p_stack.push(None) # I think this is more clear
    current_tree = expr_tree

    for i in fp_list:
        if i == "(":
            current_tree.insert_left("")
            p_stack.push(current_tree)
            current_tree = current_tree.left_child
        elif i in ["+", "-", "*", "/"]:
            current_tree.key = i
            current_tree.insert_right("")
            p_stack.push(current_tree)
            current_tree = current_tree.right_child
        elif i.isdigit():
              current_tree.key = int(i)
              parent = p_stack.pop()
              current_tree = parent
        elif i == ")":
              current_tree = p_stack.pop()

        else:
            try:
                # Convert to float (handles both '3' and '3.5')
                f = float(i)
                current_tree.set_root_val(f)
                
                # After setting a leaf node value, move back to the parent
                parent = p_stack.pop()
                current_tree = parent
            except ValueError:
                # This catches cases where 'i' isn't a number or a known operator
                raise ValueError(f"Unknown operator or operand: '{i}'")
    

    return expr_tree

In [11]:
pt = build_parse_tree("( 3 + ( 4 * 5 ) )")
pt.postorder()

3
4
5
*
+


In [12]:
pt = build_parse_tree("( ( 10 + 5 ) * 3 )")

pt.postorder()

10
5
+
3
*


<font size = "4">

Now, we write code to *evaluate* the parse tree

In [13]:
import operator

# recursive evaluation
def evaluate(parse_tree):
    operators = {
        "+": operator.add,
        "-": operator.sub,
        "*": operator.mul,
        "/": operator.truediv
    }

    left_child = parse_tree.left_child 
    right_child = parse_tree.right_child 

    if left_child and right_child: # "truthy" vs. "falsy"
        fn = operators[parse_tree.key]
        operand_1 = evaluate(left_child)
        operand_2 = evaluate(right_child)
        return fn(operand_1, operand_2)
    else:
        return parse_tree.key

In [14]:
pt = build_parse_tree("( 3 + ( 4 * 5 ) )")
print(evaluate(pt))

23


In [15]:
pt = build_parse_tree("( ( 10 + 5 ) * 3 )")
print(evaluate(pt))

45


<font size = "4">

We can also evaluate the tree by a **postorder** traversal (like reverse polish notation)

In [16]:
def postordereval(tree):
    operators = {
        "+": operator.add,
        "-": operator.sub,
        "*": operator.mul,
        "/": operator.truediv,
    }
    result_1 = None
    result_2 = None
    if tree:
        result_1 = postordereval(tree.left_child)
        result_2 = postordereval(tree.right_child)
        if result_1 and result_2:
            return operators[tree.key](result_1, result_2)
        return tree.key

pt = build_parse_tree("( ( 10 + 5 ) * 3 )")
print(postordereval(pt))

45


<font size = "4">

We can recover the original parenthesized arithmetic expression by an **inorder** traversal.

In [17]:
def print_exp(tree):
    result = ""
    if tree:
        result = "(" + print_exp(tree.left_child)
        result = result + str(tree.key)
        result = result + print_exp(tree.right_child) + ")"
    return result

In [19]:
pt = build_parse_tree("( ( 10 + 5.5 ) * 3 )")

pt_str = print_exp(pt)

print(pt_str)

(((10)+(5.5))*(3))


<font size = "4">

The parentheses around the numbers are not needed. How do we rewrite the function so they don't appear? Show me in Homework #4 !